# 🎛️ 03-自适应学习率与 Adam

> 目标：回答三个问题——**为什么逐参数步长**（稀疏特征）、**AdaGrad 为什么后期停摆**、
> **Adam 的偏差校正在干什么**。并手写 Adam 与 `torch.optim.Adam` 逐位对照。

**本讲地图**：三步演化（AdaGrad→RMSProp→Adam）→ 公式逐条拆解 → 三个实验
（稀疏回归 / 停摆演示 / 逐位对照）→ 超参与工程建议 → 面试追问 → 自测。

## §1 一刀切学习率的问题

上一讲（02 篇）已经解决了「方向震荡」——动量让更新沿历史方向平滑。
但还有一个独立的问题没有处理：**所有参数共用一个学习率 $\eta$**。

词向量、推荐系统特征等**稀疏特征**里这个问题很尖锐：
- **稀有特征**（某个商品 ID、某个用户 ID）在绝大多数样本中不出现，梯度长期为 0；
  可一旦出现，就携带巨大信息量，需要**大步长**一步修正。
- **密集特征**（偏置项、常用词向量）每个样本都出现，梯度稳定，需要**平滑的小步长**。

统一 $\eta$ 无法两全：调大则密集特征震荡发散，调小则稀有特征永远学不动。

> 🧩 **直觉**：一刀切的学习率像是「不管路况一律限速 60 公里」。
> 平坦大路（密集特征）希望开快些，崎岖山路（稀疏特征）需要小心；
> 真正的解法是每个参数有自己的「路况传感器」——根据它自己的历史梯度幅度决定步长。
> 这就是**自适应学习率**（adaptive learning rate）的思想。

## §1.2 历史背景：自适应从哪来

这条技术路线的关键节点值得记住（面试常被问「为什么先有 AdaGrad 再有 Adam」）：

1. **AdaGrad（Duchi et al., 2011）**：第一次做到「每个参数按历史梯度平方根自动缩放」。
   它把所有历史梯度平方**累加**，越少出现的参数累计值越小、步长越大——完美适配稀疏特征。
2. **RMSProp（Tieleman & Hinton, 2012，深度学习讲义）**：把「累加」换成「指数滑动平均 EMA」，
   让二阶矩有上界，解决 AdaGrad 后期停摆。
3. **Adam（Kingma & Ba, 2015）**：把 RMSProp 的二阶矩（步长）与动量的思想（方向平滑）合并，
   再加**偏差校正**解决 EMA 冷启动，从此成为深度学习默认优化器。

> 📌 记忆锚点：AdaGrad = 稀疏友好但会停摆；RMSProp = 用 EMA 救停摆但没方向记忆；
> Adam = 方向 + 步长两个统计量都做 EMA（+校正）。三步演化线非常清晰。

## §1.3 直觉：稀疏回归到底难在哪

为了让「为什么需要逐参数步长」落地，实验 1 构造了一个 **10 维稀疏线性回归**：
共 500 个样本，每个样本只激活一个特征（one-hot 风格），且 0 号特征每 10 个样本出现一次（频繁），
其余特征出现概率相同（稀疏）。

用 SGD 时所有参数共用一个 lr：频繁特征很快逼近真值，而稀疏特征的梯度绝大多数时候为 0，
几乎不动。自适应优化器则不同：
**稀疏参数的历史梯度平方和小 → 缩放后步长大 → 一次出现就能大幅修正**，
这正是 AdaGrad/RMSProp/Adam 在推荐、NLP 稀疏场景中碾压 SGD 的根本原因。

跑完实验 1 你应该看到：自适应族的收敛曲线（到真值的距离）远快于 SGD。

## §2 三步演化（背诵主线）

| 优化器 | 统计量 | 更新 | 问题 |
|--------|--------|------|------|
| AdaGrad | $G \gets G + g^2$（累加） | $w \gets w - \frac{\eta}{\sqrt{G}+\epsilon}g$ | $G$ 单调增 → 步长 $\to 0$，后期停摆 |
| RMSProp | $E \gets \beta E + (1-\beta)g^2$（EMA） | $w \gets w - \frac{\eta}{\sqrt{E}+\epsilon}g$ | 无动量，仍用固定 $\eta$ |
| Adam | $m,v$ 双 EMA + 偏差校正 | $w \gets w - \eta \frac{\hat m}{\sqrt{\hat v}+\epsilon}$ | 参数量 ×2 状态 |

**Adam 四条公式**（Kingma & Ba 2015）：

$$m_t = \beta_1 m_{t-1} + (1-\beta_1)g_t \qquad v_t = \beta_2 v_{t-1} + (1-\beta_2)g_t^2$$
$$\hat m_t = \frac{m_t}{1-\beta_1^t} \qquad \hat v_t = \frac{v_t}{1-\beta_2^t} \qquad w \gets w - \eta \frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}$$

> **为什么校正**：$m_1 = (1-\beta_1)g_1$，被 $1-\beta_1=0.1$ 缩小了 10 倍；除以 $1-\beta_1^t$ 恰好还原「真实 EMA 权重和」。（EMA 权重和为 $\sum_{i=0}^{t-1}\beta^i = (1-\beta^t)/(1-\beta)$，除以它即归一化。）

## §2.1 手写代码怎么对应公式

下面第一个代码 cell 用最简形式实现了三个优化器，统一接口 `step(w, g, t)`：
- **AdaGrad**：`self.G = self.G + g*g`（累加）→ `w - lr*g/(sqrt(G)+eps)`。
- **RMSProp**：`self.E = beta*E + (1-beta)*g*g`（EMA）→ 同样的缩放形式。
- **Adam**：`m` 是一阶矩 EMA，`v` 是二阶矩 EMA；`m_hat = m/(1-b1**t1)` 做偏差校正；
  更新式 `w - lr * m_hat/(sqrt(v_hat)+eps)`。

所有实现都取 $\epsilon=10^{-8}$ 放在分母，避免除零并给出步长下限。
请先读代码，确认每一行对应你背过的公式，再跑下面的实验。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# ---------- 手写三类自适应优化器（统一接口 step(w, g, t)） ----------
class AdaGrad:
    def __init__(self, lr=0.1, eps=1e-8):
        self.lr, self.eps = lr, eps
        self.G = None
    def step(self, w, g, t):
        self.G = g * g if self.G is None else self.G + g * g
        return w - self.lr * g / (np.sqrt(self.G) + self.eps)

class RMSProp:
    def __init__(self, lr=0.01, beta=0.9, eps=1e-8):
        self.lr, self.beta, self.eps = lr, beta, eps
        self.E = None
    def step(self, w, g, t):
        self.E = (1 - self.beta) * g * g if self.E is None else self.beta * self.E + (1 - self.beta) * g * g
        return w - self.lr * g / (np.sqrt(self.E) + self.eps)

class Adam:
    def __init__(self, lr=0.01, b1=0.9, b2=0.999, eps=1e-8):
        self.lr, self.b1, self.b2, self.eps = lr, b1, b2, eps
        self.m = self.v = None
    def step(self, w, g, t):
        t1 = t + 1
        self.m = (1 - self.b1) * g if self.m is None else self.b1 * self.m + (1 - self.b1) * g
        self.v = (1 - self.b2) * g * g if self.v is None else self.b2 * self.v + (1 - self.b2) * g * g
        m_hat = self.m / (1 - self.b1**t1)
        v_hat = self.v / (1 - self.b2**t1)
        return w - self.lr * m_hat / (np.sqrt(v_hat) + self.eps)

## §2.2 手算：偏差校正在 t=1 时到底做了什么

设 $\beta_1=0.9, \beta_2=0.999$，第一步梯度 $g_1=2$：
- 未校正：$m_1 = (1-0.9) \cdot 2 = 0.2$，$v_1 = (1-0.999) \cdot 4 = 0.004$。
  更新量 $\propto 0.2 / \sqrt{0.004} = 0.2/0.0632 \approx 3.16$（方向被缩了 10 倍）。
- 校正后：$\hat m_1 = 0.2/(1-0.9^1) = 2$，$\hat v_1 = 0.004/(1-0.999^1) = 4$。
  更新量 $\propto 2/\sqrt{4} = 1$——**恰好还原成「单位梯度应有的步长」**。

为什么重要：第 1 步的 $v$ 由 $(1-\beta_2)=0.001$ 压缩而来，极小；如果不校正，
$m/\sqrt{v}$ 的比值会被扭曲，早期更新方向失真。校正让优化器在 t 很小时也能「诚实」地使用梯度。
注意当 $t \to \infty$ 时 $1-\beta^t \to 1$，校正项趋近 1——**它只在前期起作用**。

## §2.3 实验 1：稀疏回归——怎么做、看什么

实验 1 的实现要点：
- 数据：`X` 为 500×10 的 one-hot 矩阵（每行一个 1），真权重 `w_true ~ N(0,1)`，加少量噪声。
- 全梯度：为简洁起见用**全批量梯度**（不是 mini-batch），梯度 $g = X^\top(Xw-y)/N$。
- 比较：SGD(lr=0.2) vs AdaGrad / RMSProp / Adam（lr=0.5）。
- 指标：每轮后与真值的欧氏距离 `||w - w*||`，越小越好。

跑之前先预测：谁的曲线下降最快？为什么 lr=0.5 的 AdaGrad 能比 lr=0.2 的 SGD 更稳？
（提示：自适应优化器的有效步长被二阶矩归一化，对 lr 的敏感度大幅降低。）

In [ ]:
# ---------- 实验 1：稀疏特征场景（10 维，只有 0 号特征常出现） ----------
rng = np.random.default_rng(0)
N, D = 500, 10
X = np.zeros((N, D))
for i in range(N):
    X[i, rng.integers(0, D)] = 1.0          # 每个样本只有一个特征=1（稀疏）
    if i % 10 == 0:
        X[i, 0] = 1.0                        # 0 号特征频繁出现
w_true = rng.normal(size=D)
y = X @ w_true + rng.normal(scale=0.1, size=N)

def train(opt_factory, epochs=40, lr=None):
    w = np.zeros(D); dists = [np.linalg.norm(w - w_true)]
    for e in range(epochs):
        g = X.T @ (X @ w - y) / N
        w = opt_factory(lr=lr).step(w, g, e) if lr else opt_factory().step(w, g, e)
        dists.append(np.linalg.norm(w - w_true))
    return np.array(dists)

fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.plot(train(lambda lr=None: type('S', (), {'step': lambda s, w, g, t: w - 0.2 * g})(), 40),
       label='SGD lr=0.2', color='#C44E52', lw=1.4)
ax.plot(train(lambda lr=0.5: AdaGrad(lr=lr), 40), label='AdaGrad lr=0.5', color='#DD8452', lw=1.4)
ax.plot(train(lambda lr=0.5: RMSProp(lr=lr), 40), label='RMSProp lr=0.5', color='#55A868', lw=1.4)
ax.plot(train(lambda lr=0.5: Adam(lr=lr), 40), label='Adam lr=0.5', color='#4C72B0', lw=1.4)
ax.set_xlabel('epoch'); ax.set_ylabel('||w - w*||')
ax.set_title('稀疏线性回归：自适应步长 vs SGD（自适应族更快逼近真值）', fontsize=11)
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## §2.4 实验 1 复盘：为什么自适应族更快

上图典型结果：AdaGrad / RMSProp / Adam 三条曲线在 5-10 轮内就逼近真值，
而 SGD 明显更慢甚至震荡。原因拆解：
- SGD 对所有参数用同一 lr：稀疏特征（出现率 1/10 甚至更低）梯度长期为 0，
  前几轮几乎不更新；而 0 号特征梯度大，若 lr 调大它先震荡。
- 自适应族按**每个参数的历史梯度平方**缩放：稀疏参数 $G_i$ 小 → 步长 $\eta/\sqrt{G_i}$ 大，
  一次出现就大幅修正；密集参数 $G_i$ 大 → 步长自动小，稳定前进。
- 另外注意：自适应优化器用 lr=0.5 就能稳定跑，说明它对 lr 的敏感度远低于 SGD——
  这是面试高频考点（Adam 有效步长被二阶矩归一化，约等于「每参数自适应 lr」）。

> 💡 若把 lr 调成 0.05 重跑，SGD 会几乎学不动（稀疏参数更糟），而自适应族只是稍慢——
> 这就是「自适应」带来的鲁棒性。

## §2.5 AdaGrad 后期停摆的数学直觉

AdaGrad 的累计 $G_t = \sum_{i=1}^{t} g_i^2$ **只增不减**（平方后非负）。
当训练进行到中后期，$G$ 已经很大，有效步长 $\eta/(\sqrt{G}+\epsilon) \to 0$，
参数几乎冻结——这就是「停摆」（vanishing learning rate）。

RMSProp 的修复是把累加换成 **指数滑动平均** $E_t = \beta E_{t-1} + (1-\beta) g_t^2$：
旧梯度按 $\beta^t$ 指数遗忘，$E$ 有上界（约等于近期梯度平方的均值），步长保持稳定。
这也是名字里 `Prop`（propagation）的由来——梯度信息按比例向前传播并衰减。

> ⚠️ 注意区分：AdaGrad 的 $\sqrt{G}$ 是「全部历史」的 RMS；RMSProp 的 $\sqrt{E}$ 是「近期」的 RMS。
> 一个累积、一个滑动——这是两者唯一的本质差别。

## §2.6 实验 2 预习：固定小梯度看步长演化

实验 2 用「恒定小梯度 g=0.01」这个极端情形，直观对比两种步长随时间的变化：
- AdaGrad：$G$ 每步加 $0.01^2$，$\eta/\sqrt{G}$ 单调衰减 → 曲线一路向下趋近 0。
- RMSProp：$E$ 是 EMA，很快收敛到 $0.01^2$ 附近（稳态），步长稳定在一个水平线上。

这直接可视化「停摆」：同样是 500 步，AdaGrad 步长趋 0，RMSProp 稳定。
跑之前先想一下：第二次除以的 $\epsilon=10^{-8}$ 对这种恒定梯度场景有没有影响？
（提示：$g^2=10^{-4}$，远大于 $\epsilon$，$\epsilon$ 只在梯度极小时才起作用。）

In [ ]:
# ---------- 实验 2：AdaGrad 后期停摆演示 ----------
# 固定一个小梯度 g=0.01，看步长如何随时间衰减
steps = 500
G = 0.0; lr = 1.0; eps = 1e-8
steps_adagrad = []
for t in range(steps):
    G += 0.01**2
    steps_adagrad.append(lr / (np.sqrt(G) + eps))
E = 0.0; beta = 0.9
steps_rms = []
for t in range(steps):
    E = beta * E + (1 - beta) * 0.01**2
    steps_rms.append(lr / (np.sqrt(E) + eps))

fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.plot(steps_adagrad, label='AdaGrad 步长', color='#DD8452', lw=1.5)
ax.plot(steps_rms, label='RMSProp 步长', color='#55A868', lw=1.5)
ax.set_xlabel('step'); ax.set_ylabel('有效步长 lr/(sqrt(G)+eps)')
ax.set_title('恒定小梯度下：AdaGrad 步长趋 0，RMSProp 稳定', fontsize=11)
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## §2.7 实验 2 复盘：两条曲线的解读

- **AdaGrad（橙色）**：前几步步长很大（$\sqrt{G}$ 小），随后被不断累加的 $G$ 压垮，
  到 500 步时步长只有初始值的几十分之一。真实训练中这意味着**后期几乎没有学习能力**。
- **RMSProp（绿色）**：约 10-20 步后 E 收敛到稳态 $0.01^2$，步长稳定在
  $1/\sqrt{0.0001} = 100$ 附近——注意有效步长可以远大于 lr，因为分母是梯度的 RMS。

结论：**EMA 让二阶矩既有记忆又有界**。这也解释了为什么 RMSProp/Adam 可以一路训练到底，
而 AdaGrad 只适合「短程冲刺」（或明确要稀疏解的任务，如某些在线学习场景）。

> 🧮 附：EMA 的「记忆长度」约为 $1/(1-\beta)$ 步——$\beta=0.9$ 约 10 步，
> $\beta=0.999$ 约 1000 步。这就是 Adam 的 $\beta_2=0.999$ 比 RMSProp 的 $\beta=0.9$ 大得多的原因。

## §2.8 Adam 为什么同时要 m 和 v：方向与步长解耦

RMSProp 只有二阶矩 $E$（度量梯度幅度），没有方向记忆——它仍可能沿峡谷壁振荡。
Adam 增加一阶矩 $m$（EMA of $g$）来平滑方向，两个统计量各司其职：

- $m$（一阶矩）：**方向**——把近期梯度做滑动平均，消除随机梯度噪声带来的震荡（类似动量）。
- $v$（二阶矩）：**步长**——度量该参数近期梯度的幅度，幅度大 → 除以大数 → 步长小。

两者解耦的意义：一个参数可能「方向经常变但幅度稳定」（需要 $m$ 平滑方向），
另一个可能「方向稳定但幅度忽大忽小」（需要 $v$ 缩放步长）。
Adam 用两个独立统计量分别处理，这正是它比 RMSProp（只有 $v$）更稳、
比纯动量（只有 $m$）更快适应尺度变化的原因。

> ⚠️ **易错点**：很多人以为 $m/\sqrt{v}$ 量纲是「梯度/梯度 = 无单位」，实际上两矩来自同一梯度的不同幂次，
> 比值近似给出**归一化后的更新方向**，这正是「自适应」的含义——步长与梯度尺度解耦，
> 因此 Adam 对 lr 的敏感度远低于 SGD（lr 差一个数量级通常仍能训）。

## §2.9 实现级易错点清单

1. **偏差校正遗漏**：忘除以 $1-\beta^t$，头几百步更新量异常小；实现时注意 $t$ 从 1 起计。
2. **$\epsilon$ 位置**：$1/(\sqrt{v}+\epsilon)$，不是 $1/\sqrt{v+\epsilon}$——两者数值行为不同
   （前者是「步长下限」，后者是「把 v 抬升」）。
3. **状态初始化**：`m=None, v=None` 时第一步直接取 $g$（等价于 EMA 从 0 开始），
   不要写成 `0 * g` 导致更新量为 0。
4. **状态共享**：`m/v` 必须按参数张量独立维护，不能跨参数共用一个状态。
5. **分批与全量一致**：全批量梯度（本实验）与 mini-batch 的 EMA 语义相同，
   但 mini-batch 梯度带噪声，$\beta_2$ 更大能更好抑制噪声（这也是 0.999 的由来）。

## §2.10 实验 3 预习：手写 vs torch 逐位对照

实验 3 用相同随机梯度，把**手写 Adam** 与 `torch.optim.Adam` 并排跑 20 步，
比较每个参数的最大绝对误差。若实现完全一致，误差应为 $10^{-12}$ 量级（浮点舍入）。

跑之前想三个问题：
1. torch 的 Adam 内部对 `t` 怎么计数？我们手写用 `t1 = t + 1` 对齐。
2. `betas=(0.9, 0.999), eps=1e-8` 的默认值是否与手写一致？
3. 如果误差略大于 $10^{-12}$（比如 $10^{-8}$），最可能错在哪？（提示：$\epsilon$ 位置或校正）。

这个「逐位对照」技巧在工程里非常实用：**新实现先用小规模对照官方库验证正确性**，再上大规模。

In [ ]:
# ---------- 实验 3：手写 Adam vs torch.optim.Adam 逐位对照 ----------
import torch

torch.manual_seed(1)
w_torch = torch.randn(5, 3, dtype=torch.float64, requires_grad=True)
w_np = w_torch.detach().numpy().copy()

opt_t = torch.optim.Adam([w_torch], lr=0.01, betas=(0.9, 0.999), eps=1e-8)
adam = Adam(lr=0.01, b1=0.9, b2=0.999, eps=1e-8)

maxdiff = 0.0
for t in range(20):
    # 相同的随机梯度
    g = torch.randn(5, 3, dtype=torch.float64)
    loss = (w_torch * g).sum()
    opt_t.zero_grad(); loss.backward()
    opt_t.step()
    # 手写版用同样梯度
    w_np = adam.step(w_np, g.numpy(), t)
    diff = np.abs(w_np - w_torch.detach().numpy()).max()
    maxdiff = max(maxdiff, diff)
print(f'手写 Adam vs torch.optim.Adam 最大误差: {maxdiff:.2e}')
assert maxdiff < 1e-12, '对照失败'
print('对照通过：完全一致')

## §2.11 实验 3 复盘 + 超参物理意义速查

`maxdiff < 1e-12` 证明我们的四条公式与 `torch.optim.Adam` 实现**完全同源**：
同样的 $m,v$ 递推、同样的偏差校正、同样的 $\epsilon$ 位置（$\sqrt{\hat v}+\epsilon$ 而非 $\sqrt{\hat v+\epsilon}$）。
面试被问「Adam 实现里最容易错的地方」，答案就三条：① 忘记偏差校正；② $\epsilon$ 位置写错；
③ `t` 从 1 计数还是 0 计数。

**超参数物理意义（面试背这张表）**：

| 超参 | 默认 | 物理意义 | 调大/调小后果 |
|------|------|----------|----------------|
| lr | 1e-3 | 全局步长 | 大→发散/震荡；小→慢 |
| $\beta_1$ | 0.9 | 一阶矩记忆长度（≈10 步） | 大→更平滑但滞后 |
| $\beta_2$ | 0.999 | 二阶矩记忆长度（≈1000 步） | 大→步长更稳定但反应慢 |
| $\epsilon$ | 1e-8 | 数值稳定 + 步长下限 | 大→更新更保守 |

> 💡 经验：$\beta_2$ 越大对梯度噪声越不敏感，但训练后期步长下降越慢；
> $\beta_1$ 接近 1 时动量强，适合平滑损失面；稀疏场景适当调大 $\beta_2$ 有助稀有参数。

## §3 超参与工程建议（必背）

| 优化器 | 默认 lr | 关键超参 | 适用 |
|--------|--------|----------|------|
| AdaGrad | 0.01 | — | 稀疏特征（词向量、推荐） |
| RMSProp | 0.001 | $\beta=0.9$ | RNN（Hinton 原版） |
| Adam | 0.001 | $\beta_1=0.9,\ \beta_2=0.999$ | 默认首选（NLP/RL/生成） |

- **Adam 与权重衰减耦合**：L2 正则被自适应步长干扰 → 用 AdamW（04 篇）
- **CV 上 Adam 常输给 SGD+Momentum**：平坦极小值假说（Adam 收敛到尖锐极小，泛化差）
- **Adam 不调度 lr 一般也能用**，但配 warmup + decay 更稳（05 篇）
- **首次大规模训练建议**：先 SGD/Adam 小步验证能收敛，再切换生产配置，避免调参陷阱。

## §4 常见 bug 与边界

- **偏差校正遗漏**：忘除以 $1-\beta^t$，头几百步更新量异常小（最常见的移植 bug）
- **状态共享**：m/v 必须按参数张量独立维护
- **首步 None**：`m=None` 时直接取 $g$（等价于 EMA 从 0 开始）
- **lr 忘调度**：Adam 不调度一般也能用，但配上 warmup+decay 更好（05 篇）
- **float16 下 $\epsilon$ 失效**：fp16 的 $1e-8$ 太小被下溢，需用更大 $\epsilon$ 或混合精度（07 篇）
- **梯度为 nan**：先查 lr 是否过大、是否有除零（$\sqrt{v}$ 极小、$\epsilon$ 被下溢）。

## §2.12 数学视角：自适应步长 ≈ 对角近似的牛顿法

把 Adam 的更新写成 $\Delta w = -\eta \, \mathrm{diag}(\hat v)^{-1/2} \, \hat m$——
形式上与「对角阻尼牛顿法」$\Delta w = -\eta \, H^{-1} g$ 神似（$H$ 是 Hessian）：
牛顿法用二阶导数（曲率）决定每维步长，Adam 用二阶矩 $\sqrt{v}$（梯度平方的 EMA）
近似曲率。对二次型损失，$g^2$ 的期望与 $\mathrm{diag}(H)$ 成正比，
所以自适应缩放**隐含地做了对角归一化**，这正是它对病态条件数（峡谷）比 SGD 稳的原因。

自然梯度（natural gradient）视角类似：$v$ 是 Fisher 信息矩阵对角元的近似，
除以 $\sqrt{v}$ 相当于在「参数空间度量」下走等步长。
理解这层关系后，你就明白为什么 Adam 系在 Transformer（强各向异性损失面）上表现远好于 SGD：
**每维自动归一化 ≈ 隐式的二阶信息利用**。

## §2.13 数值实现：fp16 下的 ε 陷阱（跳读 07 篇前先看这段）

混合精度训练（07 篇）里 Adam 有个经典陷阱：$\epsilon=10^{-8}$ 在 fp16 中
小于最小可表示正数（约 $6\times10^{-8}$），会被**下溢成 0**，
导致 $v + \epsilon$ 退化、除零或更新爆炸。实践中的修法：
- 把 $\epsilon$ 调大到 $10^{-6}$~$10^{-4}$；
- 或让 $m,v$ 保持 fp32（「fp32 master 权重 + fp32 优化器状态」，PyTorch AMP 的默认做法）；
- 或使用 bf16（指数范围与 fp32 相同，无下溢问题，LLM 训练主流选择）。

这也是面试「为什么 LLM 用 bf16」的答案之一：**优化器状态需要高精度**。

## §4 调参实战：一套可复制的 Adam 启动配置

第一次用 Adam 训新模型，建议的启动顺序（避免玄学调参）：
1. lr 从 $3\times10^{-4}$（小模型）或 $10^{-3}$（大模型起步）开始，
   配 warmup（前 1%-10% 步从 0 线性升到目标 lr）+ cosine 衰减（05 篇细讲）；
2. 固定 $\beta_1=0.9,\ \beta_2=0.999,\ \epsilon=10^{-8}$，只需要调 lr；
3. 若 loss 震荡：先降 lr 或加大 $\beta_2$；若后期不下降：检查 $\beta_2$ 是否过大（记忆过长）。

**为什么 Adam 要好调**：有效步长被 $\sqrt{v}$ 归一化，lr 差 3-10 倍通常都能训——
这就是「自适应」的工程红利。但要注意：**归一化不等于免费午餐**，
lr 过大仍会爆炸（$v$ 也按 $g^2$ 增长，但 EMA 有滞后），梯度裁剪仍是必需品（05 篇）。

## §5 面试追问与扩展

1. **Adam vs SGD 最终精度**：Adam 快、稳；SGD+Momentum+调度在部分 CV 任务泛化更好
2. **为什么 RNN 用 RMSProp/Adam**：RNN 梯度跨步尺度剧变，逐元素缩放能稳住（对抗梯度爆炸/消失）
3. **偏差校正的另一种等价解释**：把 EMA 归一化（除以权重和 $\sum\beta^i$）
4. **Adam 在 LLM 中的表现**：锁死 AdamW + 特殊调度（见 04/05/07 篇）
5. **AdaGrad 还能用在哪**：在线学习/稀疏特征场景反而更合适（如 FTRL 系）；
   也可用它算「频率感知」的步长下限。
6. **Adam 的第二个矩在稳定扩散/生成模型里**：v 的 EMA 平滑了每个像素的梯度尺度，
   直接关系生成质量——理解 v 的物理意义是理解自适应家族的钥匙。

## §5.2 深度追问：Adam 泛化差的「平坦极小值假说」

一个经典现象：CV 任务里 SGD+Momentum 常比 Adam 泛化好、收敛到平坦极小值，
而 Adam 常收敛到尖锐极小值。主流解释（Keskar et al. 2017 提的 batch 大小与泛化、
后续对自适应步长的分析）：
- SGD 的有效步长与梯度尺度成正比，会在**鞍点/尖锐区域**自然减速，倾向于逼近平坦盆；
- Adam 的步长被 $\sqrt{v}$ 归一化，不随梯度衰减而缩小，能**快速穿过**尖锐区域——
  但也可能停在尖锐极小值附近（损失面陡，泛化差）。

工程对策（按 ACM 面试常问）：① 最后若干 epoch 切回 SGD 微调（「SGD 收尾」）；
② 用 AdamW + 合理权重衰减抑制过拟合；③ 用 warmup/cosine 让后期步长趋 0，
相当于「以更接近 SGD 的方式收尾」。理解这个假说，才能回答「为什么我 Adam 训得好
但测试集差一截」。

## §6 延伸阅读与知识图谱

把本讲放入优化器家族谱：

```
SGD ──动量──> SGD+Momentum（02）
  \n  └─逐参步长─> AdaGrad ─EMA─> RMSProp ─+动量+校正─> Adam ─> AdamW(04)/NAdam/LAMB/Lion
```

建议下一步：
- 04 篇（AdamW 解耦权重衰减，LLM 标配）；
- 05 篇（warmup/cosine 调度、梯度裁剪/累积）；
- 理论深潜：Kingma & Ba 2015（Adam 原论文）、Loschilov & Hutter 2019（AdamW）、
  Reddi et al. 2018（AMSGrad）、Duchi et al. 2011（AdaGrad）。

>>> 下节预告：AdamW 解耦权重衰减与 LAMB / Lion（04 篇）。

## §5.9 十秒速答：面试被问「Adam 是什么」怎么答

一套可背的答法（30 秒内讲完）：
1. **一句话**：Adam 是「带动量的 RMSProp」——用一阶矩 $m$ 记方向、二阶矩 $v$ 记步长，
   都做指数滑动平均，再用偏差校正修正冷启动。
2. **公式**：$m_t=\beta_1 m_{t-1}+(1-\beta_1)g_t$；$v_t=\beta_2 v_{t-1}+(1-\beta_2)g_t^2$；
   $\hat m=m_t/(1-\beta_1^t)$，$\hat v=v_t/(1-\beta_2^t)$；更新 $w\gets w-\eta\hat m/(\sqrt{\hat v}+\epsilon)$。
3. **默认超参**：lr=1e-3，$\beta_1=0.9$，$\beta_2=0.999$，$\epsilon=1e-8$。
4. **优点**：对 lr 不敏感、逐参数自适应、收敛快；**缺点**：状态翻倍、
   与 L2 耦合（用 AdamW）、泛化有时差于 SGD+Momentum（平坦极小值假说）。
5. **何时用**：NLP/RL/生成模型默认首选；Transformer 系锁 AdamW。

把这段背熟，配合本篇两个手算例子（偏差校正、停摆），面试基本能过关。

## §6 自测清单

- [ ] 默写 AdaGrad / RMSProp / Adam 公式
- [ ] 默写 Adam 偏差校正并解释为什么
- [ ] 手写 Adam 与 torch 对照（本文已断言）
- [ ] 解释 AdaGrad 后期停摆、RMSProp 如何修复
- [ ] 说清默认超参（lr/β₁/β₂/ε）并解释物理意义
- [ ] 回答：为什么自适应优化器对 lr 不敏感
- [ ] 口算 EMA 记忆长度：β=0.9 与 β=0.999 各约多少步

> 💡 下节预告：AdamW 解耦权重衰减与 LAMB / Lion（04 篇）。